In [1]:
import matplotlib.pyplot as plt
import glob
import numpy as np

In [2]:
# 1. Buscar todos los archivos .txt en la carpeta actual
archivos = sorted(glob.glob('C:/Users/sofip/Blade/neuroscan_10_2/*.txt'))

tiempos = []
fuerzas = []
marcas_tiempo = []
fuerzas_pre_marca = []
bandera_marca_pendiente = False

print(f"Procesando {len(archivos)} archivos...")

Procesando 3 archivos...


In [3]:
%matplotlib qt
for nombre_archivo in archivos:
    with open(nombre_archivo, 'r') as f:
        lineas = f.readlines()

    for linea in lineas:
        partes = linea.strip().split(',')
        
        # Saltar líneas vacías
        if not partes:
            continue

        # CASO 1: Detectar la "marca inicial"
        # Verificamos si la segunda columna (índice 1) contiene el texto
        if len(partes) >= 2 and 'marca inicial' in partes[1]:
            bandera_marca_pendiente = True
            if ultima_fuerza_valida is not None:
                fuerzas_pre_marca.append(ultima_fuerza_valida)
            continue

        # CASO 2: Leer datos numéricos (Fuerza, Tiempo)
        # Necesitamos al menos 3 columnas: ID, Fuerza, Tiempo
        if len(partes) >= 3:
            try:
                # La fuerza es la columna 2 (índice 1)
                # El tiempo es la columna 3 (índice 2)
                fuerza = float(partes[1])
                tiempo = float(partes[2])

                tiempos.append(tiempo)
                fuerzas.append(fuerza)
                ultima_fuerza_valida = fuerza
                # Si había una marca pendiente, este es el momento donde ocurre
                if bandera_marca_pendiente:
                    marcas_tiempo.append(tiempo)
                    bandera_marca_pendiente = False
                    
            except ValueError:
                # Si falla la conversión a número, saltamos la línea
                continue

# Si no hay datos, avisar
if not tiempos:
    print("No se encontraron datos válidos.")
else:
    # Opcional: Normalizar el tiempo para que empiece en 0
    t0 = tiempos[0]
    tiempos_relativos = [t - t0 for t in tiempos]
    marcas_relativas = [m - t0 for m in marcas_tiempo]

    # Crear el gráfico
    plt.figure(figsize=(12, 6))
    
    # Graficar la señal de fuerza
    plt.plot(tiempos_relativos, fuerzas, label='Señal de Fuerza', color='blue', marker='.')

    # Dibujar las líneas verticales para cada marca encontrada
    primera_marca = True
    for marca in marcas_relativas:
        etiqueta = 'Marca Inicial' if primera_marca else ""
        plt.axvline(x=marca, color='red', linestyle='--', linewidth=1.5, label=etiqueta)
        primera_marca = False

    plt.xlabel('Tiempo (microsegundos desde inicio)')
    plt.ylabel('Fuerza')
    plt.title('Gráfico de Fuerza vs Tiempo')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Guardar y mostrar
    plt.savefig('grafico_fuerza.png')
    plt.show()

    # Boxplot de las fuerzas pre-marca ---
    if fuerzas_pre_marca:
        plt.figure(figsize=(6, 6))
        # Crear el boxplot
        plt.boxplot(fuerzas_pre_marca, vert=True, patch_artist=True, 
                   boxprops=dict(facecolor="lightblue"))
        
        plt.title(f'Fuerza justo antes de la "Marca Inicial"\n(N={len(fuerzas_pre_marca)})')
        plt.ylabel('Fuerza')
        plt.grid(True, alpha=0.3)
        plt.show()
        
        print(f"Valores de fuerza capturados pre-marca: {fuerzas_pre_marca}")
    else:
        print("No se encontraron marcas iniciales para generar el boxplot.")

Valores de fuerza capturados pre-marca: [0.0092, 0.0073, 0.0135, 0.0131, 0.007, 0.018, 0.0089, 0.0081, 0.0131, 0.0063, 0.006, 0.0052, 0.0089, 0.0064, 0.0052, 0.0131, 0.0072, 0.0065, 0.0137, 0.0074, 0.007, 0.0053, 0.007, 0.0111, 0.0102, 0.0108, 0.0062, 0.0076, 0.0052, 0.0064, 0.006, 0.0062, 0.0065, 0.0088, 0.0061, 0.0078, 0.0147, 0.0057, 0.0079, 0.0055, 0.0115, 0.0067, 0.0059, 0.0122, 0.0072, 0.0104, 0.0086, 0.0117, 0.0051, 0.0052, 0.0065, 0.0078, 0.0059, 0.0051, 0.0059, 0.0062, 0.0051, 0.0063, 0.0052, 0.0066, 0.0051]


In [22]:
def calcular_jitter_onset(tiempos, fuerzas, marcas, ventana_ruido_ms=100, n_desvios=2):
    """
    Calcula el 'Jitter' o retraso: Diferencia entre la marca del algoritmo
    y el momento real donde la señal sale del piso de ruido.
    
    Retorna:
    - jitters: Lista de diferencias de tiempo (Marca - Onset_Real) en microsegundos.
    - onsets_reales: Los tiempos exactos encontrados (para graficar).
    - indices_onsets: Los índices en el array para graficar puntitos.
    """
    jitters = []
    onsets_reales = []
    indices_onsets = []
    
    arr_t = np.array(tiempos)
    arr_f = np.array(fuerzas)
    
    # Convertir ventana a la unidad de tus tiempos (asumo microsegundos)
    ventana_ruido_us = ventana_ruido_ms * 1000 

    for t_marca in marcas:
        # 1. Encontrar índice de tu marca actual (el punto de disparo de tu algoritmo)
        idx_marca = (np.abs(arr_t - t_marca)).argmin()
        
        # 2. Definir el piso de ruido (Baseline) ANTES de la marca
        # Usamos una ventana previa para saber qué es "silencio"
        t_inicio_base = t_marca - ventana_ruido_us
        mask_base = (arr_t >= t_inicio_base) & (arr_t <= t_marca)
        datos_base = arr_f[mask_base]
        
        if len(datos_base) < 2: continue
            
        media_base = np.mean(datos_base)
        std_base = np.std(datos_base)
        
        # Si la señal es muy plana, evitamos división por cero o umbrales nulos
        if std_base == 0: std_base = 1e-6
            
        # El nivel donde consideramos que la señal se "apaga" y vuelve a ser ruido
        # Un criterio estricto es Mean + 2*SD. 
        nivel_ruido = media_base + (n_desvios * std_base)
        
        # 3. BÚSQUEDA INVERSA (BACKWARD SEARCH)
        # Recorremos desde la marca hacia atrás
        idx_onset_real = idx_marca
        encontrado = False
        
        # Retrocedemos hasta un máximo (ej. 500 puntos) para no irnos al infinito
        for i in range(idx_marca, max(0, idx_marca - 1000), -1):
            valor_actual = arr_f[i]
            
            # Si el valor cae por debajo del nivel de ruido, ahí empezó todo.
            if valor_actual <= nivel_ruido:
                idx_onset_real = i + 1 # Tomamos el punto justo antes de caer al ruido
                encontrado = True
                break
        
        if encontrado:
            t_real = arr_t[idx_onset_real]
            onsets_reales.append(t_real)
            indices_onsets.append(idx_onset_real)
            
            # El Jitter es la distancia entre tu marca y el inicio real
            # Jitter = Tiempo_Marca - Tiempo_Inicio_Real
            delta = t_marca - t_real
            jitters.append(delta)

    return jitters, onsets_reales, indices_onsets

# --- EJECUCIÓN ---

# 1. Calculamos
jitters, onsets_reales, idxs_reales = calcular_jitter_onset(tiempos, fuerzas, marcas_tiempo, n_desvios=2)

print(f"Diferencia media (Jitter): {np.mean(jitters):.2f} us")
print(f"Desviación estándar del Jitter: {np.std(jitters):.2f} us")

# 2. Graficamos para visualizar la diferencia
plt.figure(figsize=(12, 6))

# Graficamos toda la señal (o un recorte si es muy larga)
plt.plot(tiempos, fuerzas, label='Fuerza', color='lightgray')

# Graficamos TUS marcas originales
for m in marcas_tiempo:
    plt.axvline(x=m, color='red', linestyle='--', alpha=0.5, label='Tu Algoritmo' if m==marcas_tiempo[0] else "")

# Graficamos los ONSETS REALES encontrados
# Usamos los índices para sacar la fuerza correspondiente y plotear un punto
fuerzas_onsets = [fuerzas[i] for i in idxs_reales]
plt.scatter(onsets_reales, fuerzas_onsets, color='green', zorder=5, label='Inicio Real (Detectado)')

# Hacemos zoom en la primera marca para ver el detalle
if marcas_tiempo:
    centro = marcas_tiempo[10]
    #plt.xlim(centro - 200000, centro + 200000) # Zoom de +/- 20ms aprox
    plt.title("Zoom: Diferencia entre tu Marca (Roja) y el Inicio Real (Verde)")

plt.legend()
plt.ylabel('Fuerza')
plt.xlabel('Tiempo (us)')
plt.show()

# 3. Boxplot del Jitter
if jitters:
    plt.figure(figsize=(4, 6))
    plt.boxplot(jitters)
    plt.title('Distribución del Jitter\n(Retraso de tu algoritmo)')
    plt.ylabel('Tiempo (microsegundos)')
    plt.show()

Diferencia media (Jitter): 2676.89 us
Desviación estándar del Jitter: 3794.00 us
